# Expense Audit
Clean the raw expense CSV, flag problems, and summarize spend by category.

## Step 1: Imports
We import `csv` to read and write the spreadsheet files, and `Counter` to add up spend per category. Running this cell prints nothing.

In [ ]:
import csv
from collections import Counter

## Step 2: Settings
This cell stores the input file path, the two output file names, and the column names. Keeping them in one place means we only change them here. Running it prints nothing.

In [ ]:
INPUT = "input/expenses_raw.csv"
CLEAN = "expenses_clean.csv"
FLAGGED = "expenses_flagged.csv"
FIELDS = ["date", "vendor", "amount", "category"]

## Step 3: Load the raw data
This cell opens `input/expenses_raw.csv` and reads every row into a list. It prints the row count and shows the first 5 rows so we can see the messy data: odd capitalization, extra spaces, and blank dates.

In [ ]:
with open(INPUT, newline="") as f:
    rows = list(csv.DictReader(f))
print(f"Rows read: {len(rows)}")
rows[:5]

## Step 4: Normalize text
This cell trims extra spaces from vendor, category and date, and converts category names to Title Case, so "travel " and "TRAVEL" both become "Travel". It then displays the sorted list of distinct categories. We should see exactly six clean names.

In [ ]:
for row in rows:
    row["vendor"] = row["vendor"].strip()
    row["category"] = row["category"].strip().title()
    row["date"] = row["date"].strip()
sorted({r["category"] for r in rows})

## Step 5: Remove duplicates and flag missing dates
This cell loops through every row. If the same date, vendor, amount and category was already seen, it counts it as a duplicate and skips it. Rows with no date go to a flagged list, and everything else goes to the clean list. It prints the duplicates removed, rows flagged, and rows kept.

In [ ]:
seen = set()
clean, flagged, duplicates = [], [], 0
for row in rows:
    key = (row["date"], row["vendor"], row["amount"], row["category"])
    if key in seen:
        duplicates += 1
        continue
    seen.add(key)
    (clean if row["date"] else flagged).append(row)
print(f"Duplicates removed: {duplicates}")
print(f"Rows flagged (missing date): {len(flagged)}")
print(f"Rows kept: {len(clean)}")

## Step 6: Save the results
This cell writes the clean rows to `expenses_clean.csv` and the flagged rows to `expenses_flagged.csv`, each with a header line. It prints a message naming the two files it wrote. We open the clean file in LibreOffice Calc next.

In [ ]:
for path, data in ((CLEAN, clean), (FLAGGED, flagged)):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(data)
print("Wrote", CLEAN, "and", FLAGGED)

## Step 7: Total spend per category
This cell adds up the amount for each category in the clean data and prints the totals from highest to lowest. The top category is the one to review first for cost savings.

In [ ]:
totals = Counter()
for row in clean:
    totals[row["category"]] += float(row["amount"])
for category, total in totals.most_common():
    print(f"{category}: {total:.2f}")